In [1]:
import pandas as pd
import re
import os

print("Libraries loaded!")

Libraries loaded!


In [2]:
def normalize_year(year_str):
    """
    Converts messy year strings into clean YYYY-MM format.
    Examples:
        'Mar 2024' → '2024-03'
        'Dec 2022' → '2022-12'
        'TTM'      → None (we discard these)
    """
    if pd.isna(year_str):
        return None
    
    year_str = str(year_str).strip()
    
    # Remove TTM rows
    if year_str == 'TTM':
        return None
    
    # Month name to number mapping
    month_map = {
        'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04',
        'May': '05', 'Jun': '06', 'Jul': '07', 'Aug': '08',
        'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'
    }
    
    # Match formats like 'Mar 2024' or 'Dec 2022'
    match = re.match(r'([A-Za-z]{3})\s(\d{4})', year_str)
    if match:
        month = match.group(1).capitalize()
        year  = match.group(2)
        return f"{year}-{month_map.get(month, '01')}"
    
    return None

# Test it
test_values = ['Mar 2024', 'Dec 2022', 'TTM', 'Mar 2013', None]
for val in test_values:
    print(f"  '{val}' → '{normalize_year(val)}'")

  'Mar 2024' → '2024-03'
  'Dec 2022' → '2022-12'
  'TTM' → 'None'
  'Mar 2013' → '2013-03'
  'None' → 'None'


In [3]:
def normalize_ticker(ticker):
    """
    Cleans company ID (NSE ticker) to uppercase stripped format.
    Examples:
        ' tcs '   → 'TCS'
        'infy'    → 'INFY'
        'BAJAJ-AUTO' → 'BAJAJ-AUTO'
    """
    if pd.isna(ticker):
        return None
    return str(ticker).strip().upper()

# Test it
test_tickers = [' tcs ', 'infy', 'BAJAJ-AUTO', 'hdfc bank ']
for t in test_tickers:
    print(f"  '{t}' → '{normalize_ticker(t)}'")

  ' tcs ' → 'TCS'
  'infy' → 'INFY'
  'BAJAJ-AUTO' → 'BAJAJ-AUTO'
  'hdfc bank ' → 'HDFC BANK'


In [4]:
# Load raw data
pl = pd.read_excel('data/raw/profitandloss.xlsx', header=1)

print("BEFORE cleaning:")
print(f"  Rows: {len(pl)}")
print(f"  TTM rows: {(pl['year'] == 'TTM').sum()}")
print(f"  Unique years sample: {pl['year'].unique()[:5]}")

# Apply normalisers
pl['year']       = pl['year'].apply(normalize_year)
pl['company_id'] = pl['company_id'].apply(normalize_ticker)

# Drop TTM rows (they become None after normalising)
pl = pl.dropna(subset=['year'])

# Drop duplicates — keep last occurrence
pl = pl.drop_duplicates(subset=['company_id', 'year'], keep='last')

# Reset index
pl = pl.reset_index(drop=True)

print("\nAFTER cleaning:")
print(f"  Rows: {len(pl)}")
print(f"  Unique companies: {pl['company_id'].nunique()}")
print(f"  Unique years sample: {pl['year'].unique()[:5]}")
pl.head(3)

BEFORE cleaning:
  Rows: 1276
  TTM rows: 100
  Unique years sample: ['Dec 2012' 'Mar 2014' 'Mar 2015' 'Mar 2016' 'Mar 2017']

AFTER cleaning:
  Rows: 1164
  Unique companies: 100
  Unique years sample: ['2012-12' '2014-03' '2015-03' '2016-03' '2017-03']


,id,company_id,year,sales,expenses,operating_profit,opm_percentage,other_income,interest,depreciation,profit_before_tax,tax_percentage,net_profit,eps,dividend_payout
0,61,ABB,2012-12,1653,1451,202.0,12.0,33,0,19,215,33.0,145,68.0,25.0
1,62,ABB,2014-03,2276,2009,267.0,12.0,49,0,22,295,33.0,198,93.0,25.0
2,63,ABB,2015-03,2289,1977,312.0,14.0,48,0,15,344,34.0,229,108.0,29.0


In [5]:
def clean_dataframe(df):
    """
    Applies year and ticker normalisation to any dataframe
    that has company_id and year columns.
    """
    if 'company_id' in df.columns:
        df['company_id'] = df['company_id'].apply(normalize_ticker)
    
    if 'year' in df.columns:
        df['year'] = df['year'].apply(normalize_year)
        df = df.dropna(subset=['year'])
        df = df.drop_duplicates(subset=['company_id', 'year'], keep='last')
    
    return df.reset_index(drop=True)

# Load and clean all core time-series files
pl  = clean_dataframe(pd.read_excel('data/raw/profitandloss.xlsx', header=1))
bs  = clean_dataframe(pd.read_excel('data/raw/balancesheet.xlsx',  header=1))
cf  = clean_dataframe(pd.read_excel('data/raw/cashflow.xlsx',      header=1))

# Companies file — only ticker normalisation, no year column
companies = pd.read_excel('data/raw/companies.xlsx', header=1)
companies['id'] = companies['id'].apply(normalize_ticker)

print("Cleaned shapes:")
print(f"  Profit & Loss : {pl.shape}")
print(f"  Balance Sheet : {bs.shape}")
print(f"  Cash Flow     : {cf.shape}")
print(f"  Companies     : {companies.shape}")

Cleaned shapes:
  Profit & Loss : (1164, 15)
  Balance Sheet : (1165, 13)
  Cash Flow     : (1152, 7)
  Companies     : (92, 12)


In [6]:
print("Sample cleaned years from P&L:")
print(pl['year'].unique()[:10])

print("\nSample cleaned company IDs:")
print(pl['company_id'].unique()[:10])

print("\nAny TTM rows remaining?", (pl['year'] == 'TTM').sum())
print("Any null years remaining?", pl['year'].isna().sum())
print("Any null company IDs?", pl['company_id'].isna().sum())

Sample cleaned years from P&L:
['2012-12' '2014-03' '2015-03' '2016-03' '2017-03' '2018-03' '2019-03'
 '2020-03' '2021-03' '2022-03']

Sample cleaned company IDs:
['ABB' 'ADANIENSOL' 'ADANIENT' 'ADANIGREEN' 'ADANIPORTS' 'ADANIPOWER'
 'AMBUJACEM' 'APOLLOHOSP' 'ASIANPAINT' 'ATGL']

Any TTM rows remaining? 0
Any null years remaining? 0
Any null company IDs? 0


In [7]:
import sys
sys.path.append('.')

from src.etl.loader import load_all_data

data = load_all_data()

Loading all datasets...

Dataset Summary:
  Dataset                Rows   Cols
  -----------------------------------
  profitandloss          1164     15
  balancesheet           1165     13
  cashflow               1152      7
  companies                92     12
  analysis                 20      6
  documents              1585      4
  prosandcons              16      4
  sectors                  92      6
  market_cap              552      9
  financial_ratios       1184     16
  peer_groups              56      4

All datasets loaded and cleaned successfully!
